## DATA TRANSFORMATION

In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

Importamos el CSV

In [2]:
input_path = Path("data") / "staySpain_clean_22092025.pkl"
df = pd.read_pickle(input_path)

Creamos una copia del DF para hacer los cambios

In [3]:
df_transf = df.copy()

Creación columna precio por persona

In [4]:
posicion = 13
df_transf.insert(posicion, 'pricexperson', df_transf['price'] / df_transf['accommodates'])

In [5]:
df_transf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7693 entries, 0 to 7999
Data columns (total 36 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   apartment_id                 7693 non-null   int64         
 1   name                         7693 non-null   object        
 2   description                  7693 non-null   object        
 3   host_id                      7693 non-null   int64         
 4   neighbourhood_name           7693 non-null   object        
 5   neighbourhood_district       4669 non-null   object        
 6   room_type                    7693 non-null   object        
 7   accommodates                 7693 non-null   int64         
 8   bathrooms                    7650 non-null   Int64         
 9   bedrooms                     7655 non-null   Int64         
 10  beds                         7685 non-null   Int64         
 11  amenities_list               7677 non-null   obj

Transformación de columnas availabilty a positivo + porcentaje

In [6]:
posicion = 18
df_transf.insert(posicion, 'ocupation30', 30 - df_transf['availability_30'])

In [7]:
posicion = 19
df_transf.insert(posicion, 'ocup%30', (df_transf['ocupation30'] / 30) * 100)

Vamos a trabajar con la columna de disponibilidad mensual ya que el KPI se evalúa mensualmente, por lo que aunque estén creadas las columnas de porcentaje de tasa de ocupación según las diferentes variables temporales, no las vamos a utilizar.

In [8]:
posicion = 21
df_transf.insert(posicion, 'ocupation60', 60 - df_transf['availability_60'])

In [9]:
#df['ocup60%'] = (df['ocupation60'] / 60) * 100

In [10]:
posicion = 23  # Cambia según tu necesidad
df_transf.insert(posicion, 'ocupation90', 90 - df_transf['availability_90'])

In [11]:
#df['ocup90%'] = (df['ocupation90'] / 90) * 100

In [12]:
posicion = 25  # Cambia según tu necesidad
df_transf.insert(posicion, 'ocupation365', 365 - df_transf['availability_365'])

In [13]:
#df['ocup365%'] = (df['ocupation365'] / 365) * 100

In [14]:
df_transf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7693 entries, 0 to 7999
Data columns (total 41 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   apartment_id                 7693 non-null   int64         
 1   name                         7693 non-null   object        
 2   description                  7693 non-null   object        
 3   host_id                      7693 non-null   int64         
 4   neighbourhood_name           7693 non-null   object        
 5   neighbourhood_district       4669 non-null   object        
 6   room_type                    7693 non-null   object        
 7   accommodates                 7693 non-null   int64         
 8   bathrooms                    7650 non-null   Int64         
 9   bedrooms                     7655 non-null   Int64         
 10  beds                         7685 non-null   Int64         
 11  amenities_list               7677 non-null   obj

Creación de columna índice de satisfacción general

In [15]:
general_satisf = {
    'accuracy':      df_transf.review_scores_accuracy.mean(),
    'cleanliness':   df_transf.review_scores_cleanliness.mean(),
    'checkin':       df_transf.review_scores_checkin.mean(),
    'communication': df_transf.review_scores_communication.mean(),
    'location' :     df_transf.review_scores_location.mean()}

sorted_avg_scores = sorted(general_satisf.items(), key=lambda item: item[1], reverse=True)

df_transf['general_satisf'] = df_transf[
    ['review_scores_accuracy', 'review_scores_cleanliness', 
     'review_scores_checkin', 'review_scores_communication', 
     'review_scores_location']
].mean(axis=1)

posicion = 36
col_general_satisf = df_transf.pop('general_satisf')
df_transf.insert(posicion, 'general_satisf', col_general_satisf)

In [16]:
df_transf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7693 entries, 0 to 7999
Data columns (total 42 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   apartment_id                 7693 non-null   int64         
 1   name                         7693 non-null   object        
 2   description                  7693 non-null   object        
 3   host_id                      7693 non-null   int64         
 4   neighbourhood_name           7693 non-null   object        
 5   neighbourhood_district       4669 non-null   object        
 6   room_type                    7693 non-null   object        
 7   accommodates                 7693 non-null   int64         
 8   bathrooms                    7650 non-null   Int64         
 9   bedrooms                     7655 non-null   Int64         
 10  beds                         7685 non-null   Int64         
 11  amenities_list               7677 non-null   obj

In [17]:
df_transf.head(50)

,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,general_satisf,reviews_per_month,country,city,insert_date,is_instant_bookable
0,11964,A ROOM WITH A VIEW,Private bedroom in our attic apartment. Right ...,45553,Centro,NaN,Private room,2,2,1,...,100.0,100.0,100.0,100.0,100.0,75.0,Spain,Malaga,2018-07-31,False
1,21853,Bright and airy room,We have a quiet and sunny room with a good vie...,83531,C�rmenes,Latina,Private room,1,1,1,...,100.0,100.0,80.0,90.0,92.0,52.0,Spain,Madrid,2020-01-10,False
2,32347,Explore Cultural Sights from a Family-Friendly...,Open French doors and step onto a plant-filled...,139939,San Vicente,Casco Antiguo,Entire home/apt,4,1,2,...,100.0,100.0,100.0,100.0,100.0,142.0,Spain,Sevilla,2019-07-29,True
3,35379,Double 02 CasanovaRooms Barcelona,Room at a my apartment. Kitchen and 2 bathroom...,152232,l'Antiga Esquerra de l'Eixample,Eixample,Private room,2,2,1,...,100.0,100.0,100.0,90.0,98.0,306.0,Spain,Barcelona,2020-01-10,True
4,35801,Can Torras Farmhouse Studio Suite,Lay in bed & watch sunlight change the mood of...,153805,Quart,NaN,Private room,5,1,2,...,100.0,100.0,100.0,100.0,100.0,39.0,Spain,Girona,2019-02-19,False
5,48764,18th C Stone House near Costa Brava,Casa Fluvia is a charming stone village house ...,220145,Torroella de Fluvi�,NaN,Entire home/apt,8,2,4,...,100.0,100.0,90.0,100.0,98.0,27.0,Spain,Girona,2019-02-19,False
6,58512,Stylish & cozy 3BR near Sagrada Familia,Welcome to my home!<br /><br />My lovely 3 bed...,280070,el Camp de l'Arpa del Clot,Sant Mart�,Entire home/apt,6,2,3,...,90.0,90.0,90.0,90.0,90.0,329.0,Spain,Barcelona,2020-10-12,True
7,71603,PENTHOUSE1 BEST PRICE 15/21.07 PROMO LAST MINUTE!,The apartment you are about to book has everyt...,366654,la Dreta de l'Eixample,Eixample,Entire home/apt,3,2,1,...,100.0,90.0,100.0,90.0,98.0,42.0,Spain,Barcelona,2017-07-06,False
8,72150,Sunny attic duplex flat with terrace next to Sol,"The apartment is a quiet, secluded idyll in th...",364585,Embajadores,Centro,Entire home/apt,5,2,3,...,100.0,100.0,100.0,90.0,96.0,91.0,Spain,Madrid,2020-11-06,False
9,73683,Sagrada Familia area for 12 people,"An ideal location for a big group, two apartme...",135703,el Camp d'en Grassot i Gr�cia Nova,Gr�cia,Entire home/apt,12,2,4,...,100.0,100.0,90.0,90.0,94.0,14.0,Spain,Barcelona,2018-06-09,True


In [18]:
output_path = r"data/staySpain_transformed.pkl"
df_transf.to_pickle(output_path)
output_path2 = r"data/staySpain_transformed.csv"
df_transf.to_csv(output_path2)
print(f"Datos limpios guardados en: {output_path}")
print(f"Datos limpios guardados en: {output_path2}")

Datos limpios guardados en: data/staySpain_transformed.pkl
Datos limpios guardados en: data/staySpain_transformed.csv
